# 02 - Baseline Evaluation (Before Fine-Tuning)

This notebook evaluates pretrained biomedical NER models on realistic user inputs **before** we fine-tune our own model.

**Purpose:**
- Establish a baseline for comparison
- Test on real-life symptom descriptions users might input
- Understand what the model can/cannot detect out-of-the-box

After fine-tuning, we'll run the same tests to measure improvement.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers.pipelines import pipeline
import pandas as pd
from collections import defaultdict
import json
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Define Realistic User Inputs

These simulate what real users might type or say when describing their symptoms.

In [ ]:
# Realistic user inputs - simulating how people actually describe symptoms
USER_INPUTS = [
    # Simple symptom descriptions
    "I have a headache and feel tired all the time.",
    "My throat hurts and I have a runny nose.",
    "I've been having chest pain for the past 2 days.",
    "I feel dizzy when I stand up quickly.",
    "My stomach has been hurting after eating.",
    
    # More complex descriptions
    "I woke up with a fever, body aches, and chills.",
    "I have trouble sleeping and my heart races at night.",
    "My joints are swollen and painful, especially in the morning.",
    "I've noticed shortness of breath when climbing stairs.",
    "I have a persistent cough that won't go away.",
    
    # Casual/conversational style
    "Been feeling really nauseous lately, can't keep food down.",
    "My back is killing me, especially the lower back.",
    "Got this weird rash on my arm that's super itchy.",
    "Can't stop sneezing and my eyes are watery.",
    "Feeling anxious and having trouble concentrating.",
    
    # With medications mentioned
    "I took ibuprofen for my migraine but it didn't help.",
    "I've been on aspirin for my heart, but now I have stomach issues.",
    "The doctor gave me amoxicillin for my infection.",
    
    # With conditions mentioned
    "I have diabetes and lately my vision has been blurry.",
    "My asthma has been acting up, I'm wheezing a lot.",
    "I was diagnosed with hypertension last year.",
    
    # Edge cases - vague descriptions
    "I just don't feel well.",
    "Something is wrong but I can't explain it.",
    "I feel off today.",
    
    # Multiple symptoms
    "I have a high fever, severe headache, stiff neck, and sensitivity to light.",
    "Experiencing fatigue, weight loss, increased thirst, and frequent urination.",
]

print(f"Total test inputs: {len(USER_INPUTS)}")
print("\nSample inputs:")
for i, text in enumerate(USER_INPUTS[:5], 1):
    print(f"  {i}. {text}")

## 2. Load Pretrained Biomedical NER Model

We'll use `d4data/biomedical-ner-all` as our baseline - this is a commonly used pretrained biomedical NER model.

In [ ]:
# Load pretrained biomedical NER pipeline
print("Loading pretrained biomedical NER model...")
print("Model: d4data/biomedical-ner-all")

try:
    ner_pipeline = pipeline(
        "ner",
        model="d4data/biomedical-ner-all",
        aggregation_strategy="simple",
        device=0 if torch.cuda.is_available() else -1 if not torch.backends.mps.is_available() else "mps"
    )
    print("Model loaded successfully!")
    
    # Show model info
    print(f"\nModel labels: {ner_pipeline.model.config.id2label}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nTrying alternative loading method...")
    ner_pipeline = pipeline(
        "ner",
        model="d4data/biomedical-ner-all",
        aggregation_strategy="simple"
    )
    print("Model loaded!")

## 3. Run Baseline Evaluation

In [ ]:
def extract_entities(text, pipeline):
    """
    Extract entities from text using the NER pipeline.
    Returns a list of (entity_text, entity_type, confidence) tuples.
    """
    try:
        results = pipeline(text)
        entities = []
        for ent in results:
            entities.append({
                'text': ent['word'],
                'type': ent['entity_group'],
                'confidence': round(ent['score'], 3),
                'start': ent['start'],
                'end': ent['end']
            })
        return entities
    except Exception as e:
        print(f"Error processing: {text[:50]}... - {e}")
        return []

# Test on a single example first
test_text = "I have a headache and fever."
print(f"Test: {test_text}")
print(f"Entities: {extract_entities(test_text, ner_pipeline)}")

In [ ]:
# Run evaluation on all inputs
print("Running baseline evaluation on all user inputs...\n")
print("=" * 80)

results = []
entity_type_counts = defaultdict(int)
total_entities = 0

for i, text in enumerate(USER_INPUTS, 1):
    entities = extract_entities(text, ner_pipeline)
    
    result = {
        'input': text,
        'entities': entities,
        'entity_count': len(entities)
    }
    results.append(result)
    
    # Count entity types
    for ent in entities:
        entity_type_counts[ent['type']] += 1
        total_entities += 1
    
    # Print results
    print(f"\n[{i}/{len(USER_INPUTS)}] Input: {text}")
    if entities:
        for ent in entities:
            print(f"    → {ent['text']}: {ent['type']} (conf: {ent['confidence']})")
    else:
        print("    → No entities detected")

print("\n" + "=" * 80)

## 4. Analyze Baseline Results

In [ ]:
# Summary statistics
print("\n" + "=" * 50)
print("BASELINE EVALUATION SUMMARY")
print("=" * 50)

print(f"\nTotal inputs tested: {len(USER_INPUTS)}")
print(f"Total entities detected: {total_entities}")
print(f"Average entities per input: {total_entities/len(USER_INPUTS):.2f}")

# Inputs with no entities detected
no_entity_inputs = [r for r in results if r['entity_count'] == 0]
print(f"\nInputs with NO entities detected: {len(no_entity_inputs)}")
for r in no_entity_inputs:
    print(f"  - {r['input']}")

print(f"\nEntity type distribution:")
for entity_type, count in sorted(entity_type_counts.items(), key=lambda x: -x[1]):
    percentage = (count / total_entities) * 100 if total_entities > 0 else 0
    print(f"  {entity_type}: {count} ({percentage:.1f}%)")

In [ ]:
# Create a detailed results DataFrame
results_for_df = []
for r in results:
    entities_str = "; ".join([f"{e['text']} ({e['type']}, {e['confidence']})" for e in r['entities']])
    results_for_df.append({
        'Input': r['input'],
        'Entities Found': entities_str if entities_str else 'None',
        'Count': r['entity_count']
    })

df_results = pd.DataFrame(results_for_df)
print("\nDetailed Results Table:")
df_results

## 5. Identify Weaknesses (What the baseline misses)

In [ ]:
# Expected entities that a good model should detect
# We'll manually annotate what we EXPECT to find

EXPECTED_ENTITIES = {
    "I have a headache and feel tired all the time.": ["headache", "tired"],
    "My throat hurts and I have a runny nose.": ["throat hurts", "runny nose"],
    "I've been having chest pain for the past 2 days.": ["chest pain"],
    "I feel dizzy when I stand up quickly.": ["dizzy"],
    "My stomach has been hurting after eating.": ["stomach hurting"],
    "I woke up with a fever, body aches, and chills.": ["fever", "body aches", "chills"],
    "I have trouble sleeping and my heart races at night.": ["trouble sleeping", "heart races"],
    "My joints are swollen and painful, especially in the morning.": ["joints swollen", "painful"],
    "I've noticed shortness of breath when climbing stairs.": ["shortness of breath"],
    "I have a persistent cough that won't go away.": ["cough"],
    "Been feeling really nauseous lately, can't keep food down.": ["nauseous"],
    "My back is killing me, especially the lower back.": ["back pain", "lower back"],
    "Got this weird rash on my arm that's super itchy.": ["rash", "itchy"],
    "Can't stop sneezing and my eyes are watery.": ["sneezing", "watery eyes"],
    "Feeling anxious and having trouble concentrating.": ["anxious", "trouble concentrating"],
    "I took ibuprofen for my migraine but it didn't help.": ["ibuprofen", "migraine"],
    "I've been on aspirin for my heart, but now I have stomach issues.": ["aspirin", "stomach issues"],
    "The doctor gave me amoxicillin for my infection.": ["amoxicillin", "infection"],
    "I have diabetes and lately my vision has been blurry.": ["diabetes", "blurry vision"],
    "My asthma has been acting up, I'm wheezing a lot.": ["asthma", "wheezing"],
    "I was diagnosed with hypertension last year.": ["hypertension"],
}

print("Analyzing what the baseline model MISSED...\n")
print("=" * 80)

for result in results:
    text = result['input']
    if text in EXPECTED_ENTITIES:
        expected = EXPECTED_ENTITIES[text]
        detected = [e['text'].lower() for e in result['entities']]
        
        # Find missed entities (simple substring matching)
        missed = []
        for exp in expected:
            found = False
            for det in detected:
                if exp.lower() in det or det in exp.lower():
                    found = True
                    break
            if not found:
                missed.append(exp)
        
        if missed:
            print(f"\nInput: {text}")
            print(f"  Expected: {expected}")
            print(f"  Detected: {detected if detected else 'None'}")
            print(f"  MISSED: {missed}")

## 6. Calculate Baseline Metrics

In [ ]:
# Calculate approximate precision/recall based on our expectations
# Note: This is a rough estimate, not rigorous evaluation

total_expected = 0
total_detected = 0
total_correct = 0

for result in results:
    text = result['input']
    if text in EXPECTED_ENTITIES:
        expected = EXPECTED_ENTITIES[text]
        detected = [e['text'].lower() for e in result['entities']]
        
        total_expected += len(expected)
        total_detected += len(detected)
        
        # Count correct detections (fuzzy matching)
        for exp in expected:
            for det in detected:
                if exp.lower() in det or det in exp.lower():
                    total_correct += 1
                    break

# Calculate metrics
precision = total_correct / total_detected if total_detected > 0 else 0
recall = total_correct / total_expected if total_expected > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "=" * 50)
print("BASELINE METRICS (Approximate)")
print("=" * 50)
print(f"\nTotal expected entities: {total_expected}")
print(f"Total detected entities: {total_detected}")
print(f"Correctly detected: {total_correct}")
print(f"\nPrecision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1 Score: {f1:.2%}")
print("\nNote: These are approximate metrics based on fuzzy matching.")
print("After fine-tuning, we'll compare these numbers.")

## 7. Save Baseline Results for Comparison

In [ ]:
# Save baseline results to JSON for later comparison
baseline_results = {
    'timestamp': datetime.now().isoformat(),
    'model': 'd4data/biomedical-ner-all',
    'model_type': 'pretrained_baseline',
    'metrics': {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'total_inputs': len(USER_INPUTS),
        'total_entities_detected': total_entities,
        'avg_entities_per_input': total_entities / len(USER_INPUTS)
    },
    'entity_type_distribution': dict(entity_type_counts),
    'detailed_results': results
}

# Save to file
import os
os.makedirs('../results', exist_ok=True)

with open('../results/baseline_evaluation.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("Baseline results saved to results/baseline_evaluation.json")
print("\nAfter fine-tuning, run similar evaluation and compare!")

## 8. Summary & Next Steps

### Baseline Model Observations:
- The pretrained model detects some medical entities but may miss:
  - Colloquial symptom descriptions ("feeling off", "back is killing me")
  - Symptoms as general descriptions ("tired", "dizzy")
  - Some specific symptoms not in its training data

### What Fine-Tuning Will Improve:
1. Better detection of **symptoms** (our custom label)
2. Recognition of **casual/conversational** medical descriptions
3. Higher confidence scores on relevant entities
4. More consistent entity boundaries

### Next Steps:
1. Fine-tune BioBERT on our dataset with symptom labels
2. Re-run this same evaluation
3. Compare metrics (precision, recall, F1)
4. Build the inference pipeline for the app

In [ ]:
# Print final summary card
print("\n" + "#" * 60)
print("#" + " " * 58 + "#")
print("#" + "    BASELINE EVALUATION COMPLETE".center(58) + "#")
print("#" + " " * 58 + "#")
print("#" * 60)
print(f"\n  Model: d4data/biomedical-ner-all (pretrained)")
print(f"  Test inputs: {len(USER_INPUTS)}")
print(f"  Entities detected: {total_entities}")
print(f"\n  Approximate Metrics:")
print(f"    Precision: {precision:.2%}")
print(f"    Recall:    {recall:.2%}")
print(f"    F1 Score:  {f1:.2%}")
print(f"\n  Results saved to: results/baseline_evaluation.json")
print("\n" + "#" * 60)